In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="CoRal-project/coral-v2", 
    repo_type="dataset", local_dir="./coral-v2", allow_patterns="read_aloud/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 310 files: 100%|██████████| 310/310 [01:25<00:00,  3.64it/s]


'/home/ubuntu/coral-v2'

In [3]:
files = glob('coral-v2/*/*.parquet')
len(files)

310

In [4]:
df = pd.read_parquet(files[0])
df

,id_recording,id_sentence,id_speaker,text,location,location_roomdim,noise_level,noise_type,source_url,age,gender,dialect,country_birth,validated,audio,asr_prediction,asr_validation_model,asr_cer,asr_wer
0,rec_c0931098b5c1c7ce58084dbe600ea1a9,sen_00219948,spe_f9cdb0c4b4c71bc4086c6521a5416c70,De har aldrig deltaget i vinterlege,"Vibevej28, 8543 Hornslet","700,400,250",38,trafik,https://da.wikipedia.org/wiki/Marshall%C3%B8er...,61,female,østjysk,DK,0,{'bytes': b'RIFFFT\x06\x00WAVEfmt \x10\x00\x00...,de har aldrig deltaget i vinterlege,alexandrainst/coral-asr-bootstrap,0.000000,0.166667
1,rec_1f45bddc16abf76942c4f3ca9f347344,sen_00079667,spe_f9cdb0c4b4c71bc4086c6521a5416c70,Siden Gretzky forlod klubben er det ikke lykke...,"Vibevej28, 8543 Hornslet","700,400,250",38,trafik,https://da.wikipedia.org/wiki/Los%20Angeles%20...,61,female,østjysk,DK,0,{'bytes': b'RIFF\xc6\xcc\r\x00WAVEfmt \x10\x00...,siden gretzky forlod klubben er det ikke lykke...,alexandrainst/coral-asr-bootstrap,0.000000,0.117647
2,rec_fac2c3e22900009bb55145b21793a2fa,sen_00113990,spe_f9cdb0c4b4c71bc4086c6521a5416c70,Tropperne var også afhængige af de våben som b...,"Vibevej28, 8543 Hornslet","700,400,250",38,trafik,https://da.wikipedia.org/wiki/De%20R%C3%B8de%2...,61,female,østjysk,DK,0,{'bytes': b'RIFF\xc6\x07\x0f\x00WAVEfmt \x10\x...,tropperne var også afhængige af de våben som b...,alexandrainst/coral-asr-bootstrap,0.000000,0.076923
3,rec_87ab8eec45f06c851cebe23694c7d674,sen_00017274,spe_f9cdb0c4b4c71bc4086c6521a5416c70,Især lykkedes det ikke at styrke parlamentets ...,"Vibevej28, 8543 Hornslet","700,400,250",38,trafik,https://da.wikipedia.org/wiki/Tyske%20Kejserrige,61,female,østjysk,DK,0,{'bytes': b'RIFFF\x08\x07\x00WAVEfmt \x10\x00\...,især lykkedes det ikke at styrke parlamentets ...,alexandrainst/coral-asr-bootstrap,0.000000,0.125000
4,rec_13cd938cb96f4fdd3a34681712e5d4ae,sen_00073039,spe_f9cdb0c4b4c71bc4086c6521a5416c70,Altools er en række forskellelige værktøjer ud...,"Vibevej28, 8543 Hornslet","700,400,250",38,trafik,https://da.wikipedia.org/wiki/Altools,61,female,østjysk,DK,0,{'bytes': b'RIFF\xc6\xbe\x0c\x00WAVEfmt \x10\x...,altools er en række forskellige værktøjer udvi...,alexandrainst/coral-asr-bootstrap,0.028169,0.300000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843,rec_109adaa8bd0d3929cb702c8ceae91a7f,sen_00159516,spe_7b7d550e7b074dd3cc54377c67149b9c,Præsidenten skulle her efter vælges for 4 år,"Åbogade 34, 8200 Aarhus N","320,285,595",38,ingen,https://da.wikipedia.org/wiki/Dominikanske%20R...,75,male,østjysk,DK,0,{'bytes': b'RIFFF\xcd\x05\x00WAVEfmt \x10\x00\...,præsidenten skulle her vælges for fire år,alexandrainst/coral-asr-bootstrap,0.227273,0.375000
844,rec_7039d5ffa736430dd30e7461b075af82,sen_00175349,spe_7b7d550e7b074dd3cc54377c67149b9c,Det var synligt på 11 sømils afstand,"Åbogade 34, 8200 Aarhus N","320,285,595",38,ingen,https://da.wikipedia.org/wiki/Mission%20Point%...,75,male,østjysk,DK,0,{'bytes': b'RIFFF\xa0\x05\x00WAVEfmt \x10\x00\...,det var synligt på elleve sømils afstand,alexandrainst/coral-asr-bootstrap,0.166667,0.285714
845,rec_510951603565f590cd74ad022ce6dc8b,sen_00180423,spe_7b7d550e7b074dd3cc54377c67149b9c,Den strækker sig mellem Orcombe Point nær Exmo...,"Åbogade 34, 8200 Aarhus N","320,285,595",38,ingen,https://da.wikipedia.org/wiki/Jurassic%20Coast,75,male,østjysk,DK,0,{'bytes': b'RIFF\xc6\xe8\x0f\x00WAVEfmt \x10\x...,den strækker sig mellem orcombe pond exmouth i...,alexandrainst/coral-asr-bootstrap,0.214953,0.650000
846,rec_f7d05cea9147100c82e478d97de520ef,sen_00056848,spe_7b7d550e7b074dd3cc54377c67149b9c,Kraftfuldt udstyr og nye runewords blev også t...,"Åbogade 34, 8200 Aarhus N","320,285,595",38,ingen,https://da.wikipedia.org/wiki/Diablo%20II%3A%2...,75,male,østjysk,DK,0,{'bytes': b'RIFF\xc6\xff\x07\x00WAVEfmt \x10\x...,kraftfuldt udstyr og nye runewords blev også t...,alexandrainst/coral-asr-bootstrap,0.037736,0.375000


In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['id_speaker'].iloc[i]}"
            })
        
    return data

In [6]:
data = multiprocessing(files, loop, cores = 20)

100%|██████████| 10/10 [11:52<00:00, 71.29s/it]


In [7]:
len(data)

261277

In [8]:
data[0]

{'audio_filename': 'coral-v2_audio/coral-v2-read_aloud-train-00147-of-00295_0.mp3',
 'text': 'De har aldrig deltaget i vinterlege',
 'speaker': 'coral-v2_audio_spe_f9cdb0c4b4c71bc4086c6521a5416c70'}

In [9]:
with open('coral-v2.json', 'w') as fopen:
    json.dump(data, fopen)

In [10]:
audio_files = [d['audio_filename'] for d in data]

with open('coral-v2-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [11]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'coral-v2_audio/coral-v2-read_aloud-train-00147-of-00295_0.mp3',
 'text': 'De har aldrig deltaget i vinterlege',
 'speaker': 'coral-v2_audio_spe_f9cdb0c4b4c71bc4086c6521a5416c70'}

In [12]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'coral-v2')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 10.36ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  97%|█████████▋| 15.3MB / 15.8MB, 2.74MB/s  
Processing Files (1 / 1): 100%|██████████| 15.8MB / 15.8MB, 2.73MB/s  
Processing Files (1 / 1): 100%|██████████| 15.8MB / 15.8MB, 2.64MB/s  
New Data Upload: 100%|██████████| 15.8MB / 15.8MB, 2.64MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:06<00:00,  6.54s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/5dfcfbb48d96a81eb638f3f2c2f6038642e0c360', commit_message='Upload dataset', commit_description='', oid='5dfcfbb48d96a81eb638f3f2c2f6038642e0c360', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [16]:
# !zip -rq coral-v2_audio_neucodec.zip coral-v2_audio_neucodec

In [17]:
# !hf upload malaysia-ai/Multilingual-TTS coral-v2_audio_neucodec.zip --repo-type=dataset

In [20]:
# !zip -rq coral-v2_audio.zip coral-v2_audio